<a href="https://colab.research.google.com/github/vbrandaogoes-wq/vbrandaogoes-wq.github.io/blob/main/bot_alpha.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title
!pip install -q yfinance pandas numpy matplotlib

In [ ]:
# @title

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import time
from IPython.display import display, clear_output

# ==========================================
# CONFIGURAÇÕES
# ==========================================

ATIVO = "PETR4.SA"

PERIODO = "1d"
INTERVALO = "1m"

ATUALIZACAO = 15
RSI_PERIODO = 14

MAX_CANDLES = 250

em_posicao = False


# ==========================================
# CÁLCULO DO RSI
# ==========================================

def calcular_rsi(dados, periodo=14):

    variacao = dados["Close"].diff()

    ganhos = variacao.clip(lower=0)
    perdas = -variacao.clip(upper=0)

    media_ganhos = ganhos.ewm(
        alpha=1 / periodo,
        min_periods=periodo,
        adjust=False
    ).mean()

    media_perdas = perdas.ewm(
        alpha=1 / periodo,
        min_periods=periodo,
        adjust=False
    ).mean()

    rs = media_ganhos / media_perdas.replace(0, np.nan)

    rsi = 100 - (100 / (1 + rs))

    rsi = rsi.mask(
        (media_perdas == 0) & (media_ganhos > 0),
        100
    )

    rsi = rsi.mask(
        (media_perdas == 0) & (media_ganhos == 0),
        50
    )

    return rsi


# ==========================================
# ANÁLISE DOS INDICADORES
# ==========================================

def analisar(dados):

    dados = dados.copy()

    dados["MM9"] = dados["Close"].rolling(9).mean()
    dados["MM21"] = dados["Close"].rolling(21).mean()

    dados["RSI"] = calcular_rsi(
        dados,
        RSI_PERIODO
    )

    anterior = dados.iloc[-2]
    atual = dados.iloc[-1]

    cruzamento_compra = (
        anterior["MM9"] <= anterior["MM21"]
        and atual["MM9"] > atual["MM21"]
    )

    cruzamento_venda = (
        anterior["MM9"] >= anterior["MM21"]
        and atual["MM9"] < atual["MM21"]
    )

    compra = (
        cruzamento_compra
        and 50 < atual["RSI"] < 70
    )

    venda = (
        cruzamento_venda
        or (
            anterior["RSI"] >= 50
            and atual["RSI"] < 50
        )
    )

    return dados, compra, venda


# ==========================================
# BUSCAR DADOS
# ==========================================

def buscar_dados():

    dados = yf.download(
        ATIVO,
        period=PERIODO,
        interval=INTERVALO,
        progress=False,
        auto_adjust=False,
        prepost=False,
        timeout=20
    )

    if dados.empty:
        return None

    # Corrigir MultiIndex

    if isinstance(dados.columns, pd.MultiIndex):

        if ATIVO in dados.columns.get_level_values(-1):

            dados = dados.xs(
                ATIVO,
                axis=1,
                level=-1
            )

        else:

            dados.columns = dados.columns.get_level_values(0)

    # Ajustar horário para Brasília

    if dados.index.tz is not None:

        dados.index = dados.index.tz_convert(
            "America/Sao_Paulo"
        ).tz_localize(None)

    # Remover duplicatas e ordenar

    dados = dados[
        ~dados.index.duplicated(keep="last")
    ]

    dados = dados.sort_index()

    # Manter somente os candles de hoje

    hoje = pd.Timestamp.now(
        tz="America/Sao_Paulo"
    ).tz_localize(None).normalize()

    dados = dados[dados.index >= hoje]

    # Remover dados incompletos

    dados = dados.dropna(
        subset=["Open", "High", "Low", "Close"]
    )

    if len(dados) < 30:
        return None

    return dados


# ==========================================
# CONSTRUIR GRÁFICO
# ==========================================

def desenhar_grafico(dados):

    visual = dados.tail(MAX_CANDLES).copy()

    fig, (ax_preco, ax_volume, ax_rsi) = plt.subplots(
        3,
        1,
        figsize=(15, 10),
        sharex=True,
        gridspec_kw={
            "height_ratios": [5, 1.5, 2]
        }
    )

    x = np.arange(len(visual))

    # --------------------------------------
    # CANDLES
    # --------------------------------------

    for i, (_, candle) in enumerate(visual.iterrows()):

        abertura = float(candle["Open"])
        fechamento = float(candle["Close"])
        maxima = float(candle["High"])
        minima = float(candle["Low"])

        cor = (
            "green"
            if fechamento >= abertura
            else "red"
        )

        ax_preco.vlines(
            i,
            minima,
            maxima,
            color=cor,
            linewidth=1
        )

        corpo = plt.Rectangle(
            (
                i - 0.3,
                min(abertura, fechamento)
            ),
            0.6,
            max(abs(fechamento - abertura), 0.001),
            facecolor=cor,
            edgecolor=cor
        )

        ax_preco.add_patch(corpo)

    # --------------------------------------
    # MÉDIAS MÓVEIS
    # --------------------------------------

    ax_preco.plot(
        x,
        visual["MM9"],
        color="blue",
        linewidth=1.3,
        label="MM9"
    )

    ax_preco.plot(
        x,
        visual["MM21"],
        color="orange",
        linewidth=1.3,
        label="MM21"
    )

    ax_preco.set_title(
        f"{ATIVO} - Análise técnica do dia",
        fontsize=15
    )

    ax_preco.set_ylabel("Preço (R$)")

    ax_preco.legend(loc="upper left")

    ax_preco.grid(alpha=0.2)

    ax_preco.set_xlim(-1, len(visual))

    # --------------------------------------
    # VOLUME
    # --------------------------------------

    cores = [
        "green" if c >= o else "red"
        for c, o in zip(
            visual["Close"],
            visual["Open"]
        )
    ]

    ax_volume.bar(
        x,
        visual["Volume"].fillna(0),
        color=cores,
        width=0.6
    )

    ax_volume.set_ylabel("Volume")

    ax_volume.grid(alpha=0.2)

    # --------------------------------------
    # RSI
    # --------------------------------------

    ax_rsi.plot(
        x,
        visual["RSI"],
        color="purple",
        linewidth=1.4,
        label="RSI 14"
    )

    ax_rsi.axhline(
        70,
        color="red",
        linestyle="--"
    )

    ax_rsi.axhline(
        30,
        color="green",
        linestyle="--"
    )

    ax_rsi.axhline(
        50,
        color="gray",
        linestyle=":"
    )

    ax_rsi.set_ylim(0, 100)

    ax_rsi.set_ylabel("RSI")

    ax_rsi.grid(alpha=0.2)

    # --------------------------------------
    # HORÁRIOS
    # --------------------------------------

    intervalo = max(1, len(visual) // 10)

    posicoes = list(range(0, len(visual), intervalo))

    rotulos = [
        visual.index[i].strftime("%H:%M")
        for i in posicoes
    ]

    ax_rsi.set_xticks(posicoes)

    ax_rsi.set_xticklabels(
        rotulos,
        rotation=30
    )

    ax_rsi.set_xlabel("Horário de Brasília")

    plt.tight_layout()

    return fig


# ==========================================
# MONITORAMENTO
# ==========================================

print("Iniciando analisador de PETR4...")

try:

    while True:

        try:

            dados = buscar_dados()

            if dados is None:

                clear_output(wait=True)

                print("Dados insuficientes.")
                print("Aguardando mais candles...")

                time.sleep(ATUALIZACAO)
                continue

            dados, compra, venda = analisar(dados)

            atual = dados.iloc[-1]

            # ----------------------------------
            # VERIFICAR INDICADORES
            # ----------------------------------

            if pd.isna(atual["RSI"]):

                print("Aguardando indicadores...")

                time.sleep(ATUALIZACAO)
                continue

            # ----------------------------------
            # SINAIS
            # ----------------------------------

            sinal = "AGUARDAR"

            if compra and not em_posicao:

                sinal = "COMPRA"
                em_posicao = True

            elif venda and em_posicao:

                sinal = "VENDA"
                em_posicao = False

            # ----------------------------------
            # GRÁFICO
            # ----------------------------------

            fig = desenhar_grafico(dados)

            clear_output(wait=True)

            display(fig)

            plt.close(fig)

            # ----------------------------------
            # INFORMAÇÕES
            # ----------------------------------

            print("=" * 45)
            print("       ANALISADOR DE PETR4")
            print("=" * 45)

            print(f"Horário: {dados.index[-1]}")
            print(f"Preço: R$ {atual['Close']:.2f}")

            print(f"RSI: {atual['RSI']:.2f}")

            print(f"MM9: R$ {atual['MM9']:.2f}")
            print(f"MM21: R$ {atual['MM21']:.2f}")

            print("-" * 45)

            print(f"SINAL: {sinal}")

            print(
                "Posição:",
                "COMPRADO" if em_posicao
                else "FORA DO MERCADO"
            )

            print("=" * 45)

            print(
                f"\nÚltima atualização: {dados.index[-1]}"
            )

            print(
                f"Próxima atualização em {ATUALIZACAO} segundos."
            )

        except Exception as erro:

            clear_output(wait=True)

            print("Erro na atualização:", erro)

        time.sleep(ATUALIZACAO)

except KeyboardInterrupt:

    print("\nMonitoramento encerrado.")

Dados insuficientes.
Aguardando mais candles...
